# IMPORTS

In [1]:
import pandas as pd
import os

# INSPECTION

In [2]:
# Defining file paths in a dictionary for easy iteration
file_paths = {
    "Transactions": r"D:\Data_Science\Projects\Portfolio_Projects\marketing-campaign-attribution-dashboard\dataset\raw\transactions.csv",
    "Campaigns": r"D:\Data_Science\Projects\Portfolio_Projects\marketing-campaign-attribution-dashboard\dataset\raw\campaigns.csv",
    "Customers": r"D:\Data_Science\Projects\Portfolio_Projects\marketing-campaign-attribution-dashboard\dataset\raw\customers.csv",
    "Events": r"D:\Data_Science\Projects\Portfolio_Projects\marketing-campaign-attribution-dashboard\dataset\raw\events.csv",
    "Products": r"D:\Data_Science\Projects\Portfolio_Projects\marketing-campaign-attribution-dashboard\dataset\raw\products.csv"
}

def inspect_data(name, path):
    print(f"\n{'='*20} INSPECTING: {name} {'='*20}")
    
    # Loading the data
    df = pd.read_csv(path)
    
    # 1. Basic Dimensions
    print(f"--- Shape (Rows, Columns): {df.shape}")
    
    # 2. Column Names & Data Types
    print("\n--- Info & Data Types:")
    print(df.info())
    
    # 3. Missing Values
    print("\n--- Missing Values per Column:")
    print(df.isnull().sum())
    
    # 4. Duplicates
    duplicate_count = df.duplicated().sum()
    print(f"\n--- Total Duplicate Rows: {duplicate_count}")
    
    # 5. Statistical Summary (Numerical columns)
    print("\n--- Descriptive Statistics:")
    print(df.describe())
    
    # 6. Sample Data
    print("\n--- First 5 Rows:")
    print(df.head())
    
    return df

# Executing inspection for all files
dfs = {}
for name, path in file_paths.items():
    try:
        dfs[name] = inspect_data(name, path)
    except Exception as e:
        print(f"Error loading {name}: {e}")

print("\n\n✅ Inspection Complete.")


==================== INSPECTING: Transactions ====================
--- Shape (Rows, Columns): (103127, 9)

--- Info & Data Types:
<class 'pandas.DataFrame'>
RangeIndex: 103127 entries, 0 to 103126
Data columns (total 9 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   transaction_id    103127 non-null  int64  
 1   timestamp         103127 non-null  str    
 2   customer_id       103127 non-null  int64  
 3   product_id        92678 non-null   float64
 4   quantity          103127 non-null  int64  
 5   discount_applied  103127 non-null  float64
 6   gross_revenue     92678 non-null   float64
 7   campaign_id       103127 non-null  int64  
 8   refund_flag       103127 non-null  int64  
dtypes: float64(3), int64(5), str(1)
memory usage: 7.1 MB
None

--- Missing Values per Column:
transaction_id          0
timestamp               0
customer_id             0
product_id          10449
quantity                0
discount_applied 

# CLEANING

In [3]:
# The Transactions Cleaning

# Creating a working copy of the Transactions dataframe
df_transactions = dfs["Transactions"].copy()

print(f"Original shape: {df_transactions.shape}")

# --- 1. Handling Missing Values ---
df_transactions.dropna(subset=['product_id', 'gross_revenue'], inplace=True)

# --- 2. Fixing Data Types ---
# Converting 'timestamp' from text to datetime format
df_transactions['timestamp'] = pd.to_datetime(df_transactions['timestamp'])

# Converting 'product_id' back to integer since NaNs are removed
df_transactions['product_id'] = df_transactions['product_id'].astype(int)

# --- 3. Verifying the Cleaning ---
print(f"Cleaned shape: {df_transactions.shape}")
print("\n--- Updated Data Types ---")
print(df_transactions.dtypes)
print("\n--- Remaining Missing Values ---")
print(df_transactions.isnull().sum())
df_transactions.head()

Original shape: (103127, 9)
Cleaned shape: (92678, 9)

--- Updated Data Types ---
transaction_id               int64
timestamp           datetime64[us]
customer_id                  int64
product_id                   int64
quantity                     int64
discount_applied           float64
gross_revenue              float64
campaign_id                  int64
refund_flag                  int64
dtype: object

--- Remaining Missing Values ---
transaction_id      0
timestamp           0
customer_id         0
product_id          0
quantity            0
discount_applied    0
gross_revenue       0
campaign_id         0
refund_flag         0
dtype: int64


,transaction_id,timestamp,customer_id,product_id,quantity,discount_applied,gross_revenue,campaign_id,refund_flag
0,1,2021-12-27 08:25:15,59540,1630,3,0.00,43.74,0,0
1,2,2023-06-06 21:14:26,54871,1901,3,0.00,174.78,21,0
2,3,2023-08-31 05:29:54,51818,1884,1,0.00,40.61,37,0
3,4,2022-06-26 20:33:46,18164,1114,2,0.15,68.76,13,0
4,5,2023-07-26 18:12:35,86915,408,1,0.00,14.64,4,0


In [4]:
# The Campaigns Cleaning

# Creating a working copy of the Campaigns dataframe
df_campaigns = dfs["Campaigns"].copy()

print(f"Original shape: {df_campaigns.shape}")

# --- 1. Fixing Data Types ---
# Converting 'start_date' and 'end_date' from text strings to proper datetime objects
df_campaigns['start_date'] = pd.to_datetime(df_campaigns['start_date'])
df_campaigns['end_date'] = pd.to_datetime(df_campaigns['end_date'])

# --- 2. Calculating Campaign Duration ---
# Adding an engineered feature
df_campaigns['campaign_duration_days'] = (df_campaigns['end_date'] - df_campaigns['start_date']).dt.days

# --- 3. Verifying the Cleaning ---
print(f"Cleaned shape: {df_campaigns.shape}")
print("\n--- Updated Data Types ---")
print(df_campaigns.dtypes)
print("\n--- First 5 Rows (Checking new Duration column) ---")
df_campaigns.head()

Original shape: (50, 7)
Cleaned shape: (50, 8)

--- Updated Data Types ---
campaign_id                        int64
channel                              str
objective                            str
start_date                datetime64[us]
end_date                  datetime64[us]
target_segment                       str
expected_uplift                  float64
campaign_duration_days             int64
dtype: object

--- First 5 Rows (Checking new Duration column) ---


,campaign_id,channel,objective,start_date,end_date,target_segment,expected_uplift,campaign_duration_days
0,1,Paid Search,Cross-sell,2021-10-25,2021-11-26,Deal Seekers,0.022,32
1,2,Email,Retention,2021-10-24,2021-12-24,Deal Seekers,0.116,61
2,3,Email,Reactivation,2023-10-08,2023-11-30,Churn Risk,0.100,53
3,4,Display,Reactivation,2022-07-25,2022-10-07,Deal Seekers,0.111,74
4,5,Social,Acquisition,2022-07-09,2022-09-29,New Customers,0.144,82


In [5]:
# The Customers Cleaning

# Creating a working copy of the Customers dataframe
df_customers = dfs["Customers"].copy()

print(f"Original shape: {df_customers.shape}")

# --- 1. Fixing Data Types ---
# Converting 'signup_date' from text string to proper datetime object
df_customers['signup_date'] = pd.to_datetime(df_customers['signup_date'])

# --- 2. Feature Engineering: Age Groups ---
# Creating bins for standard marketing demographics.
bins = [17, 24, 34, 44, 54, 64, 100]
labels = ['18-24', '25-34', '35-44', '45-54', '55-64', '65+']
df_customers['age_group'] = pd.cut(df_customers['age'], bins=bins, labels=labels)

# --- 3. Verifying the Cleaning ---
print(f"Cleaned shape: {df_customers.shape}")
print("\n--- Updated Data Types ---")
print(df_customers.dtypes)
print("\n--- First 5 Rows (Checking new Age Group column) ---")
df_customers.head()

Original shape: (100000, 7)
Cleaned shape: (100000, 8)

--- Updated Data Types ---
customer_id                     int64
signup_date            datetime64[us]
country                           str
age                             int64
gender                            str
loyalty_tier                      str
acquisition_channel               str
age_group                    category
dtype: object

--- First 5 Rows (Checking new Age Group column) ---


,customer_id,signup_date,country,age,gender,loyalty_tier,acquisition_channel,age_group
0,1,2021-04-08,BR,48,Male,Bronze,Referral,45-54
1,2,2023-04-28,IN,36,Female,Silver,Organic,35-44
2,3,2022-12-18,UK,35,Female,Silver,Organic,35-44
3,4,2022-04-26,US,45,Male,Silver,Paid Search,45-54
4,5,2022-04-20,IN,53,Male,Silver,Organic,45-54


In [ ]:
# Creating a working copy of the Events dataframe
df_events = dfs["Events"].copy()

print(f"Original shape: {df_events.shape}")

# --- 1. Handling Missing Values Strategically ---
# Missing device types are bucketed into an 'Unknown' category
df_events['device_type'] = df_events['device_type'].fillna('Unknown')

# Filling NaNs with 0 (since real product IDs start at 1) to preserve these events for funnel tracking.
df_events['product_id'] = df_events['product_id'].fillna(0)

# --- 2. Fixing Data Types ---
# Converting 'timestamp' from text to datetime format
df_events['timestamp'] = pd.to_datetime(df_events['timestamp'])

# Converting 'product_id' to integer now that NaNs are filled
df_events['product_id'] = df_events['product_id'].astype(int)

# --- 3. Verifying the Cleaning ---
print(f"Cleaned shape: {df_events.shape}")
print("\n--- Updated Data Types ---")
print(df_events.dtypes)
print("\n--- Remaining Missing Values ---")
print(df_events.isnull().sum())
df_events.head()

Original shape: (2000000, 12)
Cleaned shape: (2000000, 12)

--- Updated Data Types ---
event_id                         int64
timestamp               datetime64[us]
customer_id                      int64
session_id                       int64
event_type                         str
product_id                       int64
device_type                        str
traffic_source                     str
campaign_id                      int64
page_category                      str
session_duration_sec           float64
experiment_group                   str
dtype: object

--- Remaining Missing Values ---
event_id                0
timestamp               0
customer_id             0
session_id              0
event_type              0
product_id              0
device_type             0
traffic_source          0
campaign_id             0
page_category           0
session_duration_sec    0
experiment_group        0
dtype: int64


,event_id,timestamp,customer_id,session_id,event_type,product_id,device_type,traffic_source,campaign_id,page_category,session_duration_sec,experiment_group
0,1,2021-01-14 13:35:43,43812,535101,view,1004,desktop,Email,43,PLP,115.1,Control
1,2,2021-12-03 21:36:50,71340,96426,add_to_cart,986,desktop,Email,10,PDP,32.4,Variant_A
2,3,2021-12-27 08:25:15,59540,220126,purchase,1630,mobile,Organic,0,PDP,190.7,Variant_A
3,4,2022-01-22 15:06:54,3601,484555,add_to_cart,1532,desktop,Paid Search,30,Checkout,134.8,Variant_B
4,5,2021-05-10 12:03:09,92735,60646,bounce,0,desktop,Email,26,PLP,53.1,Variant_A


In [7]:
# Creating a working copy of the Products dataframe
df_products = dfs["Products"].copy()

print(f"Original shape: {df_products.shape}")

# --- 1. Fixing Data Types ---
# Converting 'launch_date' from text string to proper datetime object
df_products['launch_date'] = pd.to_datetime(df_products['launch_date'])

# --- 2. Feature Engineering: Price Tiers ---
# Categorizing products by base price to make dashboard filtering easier.
# Based on the statistics: 25% < $32, 75% < $98
price_bins = [0, 35, 100, 500]
price_labels = ['Budget (<$35)', 'Mid-Range ($35-$100)', 'Premium (>$100)']
df_products['price_tier'] = pd.cut(df_products['base_price'], bins=price_bins, labels=price_labels)

# --- 3. Verifying the Cleaning ---
print(f"Cleaned shape: {df_products.shape}")
print("\n--- Updated Data Types ---")
print(df_products.dtypes)
print("\n--- First 5 Rows (Checking new Price Tier column) ---")
df_products.head()

Original shape: (2000, 6)
Cleaned shape: (2000, 7)

--- Updated Data Types ---
product_id              int64
category                  str
brand                     str
base_price            float64
launch_date    datetime64[us]
is_premium              int64
price_tier           category
dtype: object

--- First 5 Rows (Checking new Price Tier column) ---


,product_id,category,brand,base_price,launch_date,is_premium,price_tier
0,1,Grocery,Brand_58,14.19,2021-08-02,0,Budget (<$35)
1,2,Fashion,Brand_1,25.80,2021-09-14,0,Budget (<$35)
2,3,Electronics,Brand_70,165.46,2021-01-18,1,Premium (>$100)
3,4,Fashion,Brand_56,75.45,2023-03-03,1,Mid-Range ($35-$100)
4,5,Sports,Brand_1,72.50,2022-04-19,1,Mid-Range ($35-$100)


In [8]:
# --- Reordering Engineered Columns ---

# 1. Campaigns: Placing duration next to the dates
df_campaigns = df_campaigns[[
    'campaign_id', 'channel', 'objective', 
    'start_date', 'end_date', 'campaign_duration_days', 
    'target_segment', 'expected_uplift'
]]

# 2. Customers: Placing age_group next to age
df_customers = df_customers[[
    'customer_id', 'signup_date', 'country', 
    'age', 'age_group', 
    'gender', 'loyalty_tier', 'acquisition_channel'
]]

# 3. Products: Placing price_tier next to base_price
df_products = df_products[[
    'product_id', 'category', 'brand', 
    'base_price', 'price_tier', 
    'launch_date', 'is_premium'
]]

# --- Verifying the Reorder ---
print("Campaigns:", df_campaigns.columns.tolist())
print("Customers:", df_customers.columns.tolist())
print("Products: ", df_products.columns.tolist())

Campaigns: ['campaign_id', 'channel', 'objective', 'start_date', 'end_date', 'campaign_duration_days', 'target_segment', 'expected_uplift']
Customers: ['customer_id', 'signup_date', 'country', 'age', 'age_group', 'gender', 'loyalty_tier', 'acquisition_channel']
Products:  ['product_id', 'category', 'brand', 'base_price', 'price_tier', 'launch_date', 'is_premium']


# EXPORTS

In [11]:
# Defining the output directory
output_dir = r"D:\Data_Science\Projects\Portfolio_Projects\marketing-campaign-attribution-dashboard\dataset\processed"

# Ensuring the directory exists just in case
os.makedirs(output_dir, exist_ok=True)

print("Exporting cleaned datasets to processed folder...\n")

# Exporting each dataframe without the index column
df_transactions.to_csv(os.path.join(output_dir, "transactions_cleaned.csv"), index=False)
print("transactions_cleaned.csv saved.")

df_campaigns.to_csv(os.path.join(output_dir, "campaigns_cleaned.csv"), index=False)
print("campaigns_cleaned.csv saved.")

df_customers.to_csv(os.path.join(output_dir, "customers_cleaned.csv"), index=False)
print("customers_cleaned.csv saved.")

df_events.to_csv(os.path.join(output_dir, "events_cleaned.csv"), index=False)
print("events_cleaned.csv saved.")

df_products.to_csv(os.path.join(output_dir, "products_cleaned.csv"), index=False)
print("products_cleaned.csv saved.")

print("\nAll files successfully exported to the processed directory!")

Exporting cleaned datasets to processed folder...

transactions_cleaned.csv saved.
campaigns_cleaned.csv saved.
customers_cleaned.csv saved.
events_cleaned.csv saved.
products_cleaned.csv saved.

All files successfully exported to the processed directory!
